In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from matplotlib.patches import Patch
import warnings

warnings.filterwarnings('ignore')


# ============================================================
# GLOBAL STYLING FOR THESIS PRESENTATION
# LARGE, CLEAR AND READABLE AFTER POWERPOINT RESIZING
# ============================================================

plt.rcParams['font.size'] = 20

# Axis titles
plt.rcParams['axes.labelsize'] = 24
plt.rcParams['axes.labelweight'] = 'bold'

# Graph titles
plt.rcParams['axes.titlesize'] = 26
plt.rcParams['axes.titleweight'] = 'bold'

# Tick labels
plt.rcParams['xtick.labelsize'] = 20
plt.rcParams['ytick.labelsize'] = 20

# Legends
plt.rcParams['legend.fontsize'] = 20

# Figure titles
plt.rcParams['figure.titlesize'] = 28

# High quality rendering (600 DPI requested)
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 600

# ============================================================
# PRESENTATION CONSTANTS
# ============================================================

AXIS_LABEL_SIZE = 24
TICK_LABEL_SIZE = 20
TITLE_SIZE = 26
LEGEND_SIZE = 20
VALUE_LABEL_SIZE = 16

FIG_SINGLE = (12, 7)
FIG_COMBINED = (14, 8)
FIG_TWO_PANEL = (18, 8)


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def format_axis(ax, xlabel=None, ylabel=None, title=None):
    """
    Apply consistent presentation formatting to an axis.
    """

    if xlabel is not None:
        ax.set_xlabel(
            xlabel,
            fontsize=AXIS_LABEL_SIZE,
            fontweight='bold',
            labelpad=10
        )

    if ylabel is not None:
        ax.set_ylabel(
            ylabel,
            fontsize=AXIS_LABEL_SIZE,
            fontweight='bold',
            labelpad=12
        )

    if title is not None:
        ax.set_title(
            title,
            fontsize=TITLE_SIZE,
            fontweight='bold',
            pad=15
        )

    ax.tick_params(
        axis='both',
        which='major',
        labelsize=TICK_LABEL_SIZE,
        width=1.5,
        length=6
    )

    for label in ax.get_xticklabels():
        label.set_fontweight('bold')

    for label in ax.get_yticklabels():
        label.set_fontweight('bold')


def save_figure(fig, filename):
    """
    Save figures at high resolution (600 DPI) with enough space
    around axis titles and labels.
    """

    fig.savefig(
        filename,
        dpi=600,
        bbox_inches='tight',
        pad_inches=0.25,
        facecolor='white'
    )

    plt.close(fig)


def add_value_labels(
    ax,
    bars,
    fmt='{:.2f}',
    fontsize=VALUE_LABEL_SIZE
):
    """
    Add numerical values above bars using a proportional offset 
    so small values (like CV) don't float too high.
    """

    for bar in bars:

        height = bar.get_height()

        if height is None or np.isnan(height):
            continue

        # Proportional offset scales naturally with bar height
        offset = 0.03 * abs(height) + 0.05

        ax.text(
            bar.get_x() + bar.get_width() / 2.,
            height + offset,
            fmt.format(height),
            ha='center',
            va='bottom',
            fontsize=fontsize,
            fontweight='bold'
        )


# ============================================================
# 1. FILE NAMES
# ============================================================

file_500 = "500 DEFOCUS1 STEPS.csv"
file_1000 = "1000 DEFOCUS STEPS.csv"
file_1500 = "1500  DEFOCUS STEPS.csv"

out_dir = 'autofocus_plots_presentation_ready46'

os.makedirs(out_dir, exist_ok=True)


# ============================================================
# 2. CUSTOM CSV PARSER
# ============================================================

def parse_file_updated(filename, defocus):

    df = pd.read_csv(filename)

    headers = df.iloc[0].values

    parsed_data = []

    current_method = "SVR"

    for idx, row in df.iloc[1:].iterrows():

        val0 = str(row.iloc[0]).strip().upper()

        if val0 in [
            "SVR",
            "RANDOM FOREST",
            "CUSTOM",
            "ANN",
            "INBUILT"
        ]:
            current_method = val0
            continue

        if pd.isna(row.iloc[0]) or val0 == 'NAN':
            continue

        row_dict = {
            "Method": current_method,
            "Defocus": defocus
        }

        for h, v in zip(headers, row.values):

            if pd.notna(h) and str(h).strip() != "":

                h_clean = (
                    str(h)
                    .strip()
                    .replace(" ", "_")
                    .upper()
                )

                row_dict[h_clean] = v

        parsed_data.append(row_dict)

    df_clean = pd.DataFrame(parsed_data)

    col_mapping = {
        'Z_FINAL': 'Z_Final',
        'INITIAL_VARIANCE': 'Initial_Variance',
        'FINAL_VARIANCE': 'Final_Variance',
        'AUTOFOCUS_TIME': 'Autofocus_Time',
        'CPU': 'CPU',
        'RAM': 'RAM'
    }

    df_clean.rename(
        columns=col_mapping,
        inplace=True
    )

    numeric_cols = [
        'Z_Final',
        'Initial_Variance',
        'Final_Variance',
        'Autofocus_Time',
        'CPU',
        'RAM'
    ]

    for c in numeric_cols:

        if c in df_clean.columns:

            df_clean[c] = pd.to_numeric(
                df_clean[c],
                errors='coerce'
            )

    return df_clean


# ============================================================
# 3. LOAD AND COMBINE DATASETS
# ============================================================

df_500 = parse_file_updated(file_500, 500)
df_1000 = parse_file_updated(file_1000, 1000)
df_1500 = parse_file_updated(file_1500, 1500)

df_all = pd.concat(
    [
        df_500,
        df_1000,
        df_1500
    ],
    ignore_index=True
)


# ============================================================
# 4. COMPUTE METRICS
# ============================================================

methods = [
    'SVR',
    'RANDOM FOREST',
    'CUSTOM',
    'INBUILT'
]

defocus_values = [
    500,
    1000,
    1500
]

all_metrics = []

for defocus in defocus_values:

    df_defocus = (
        df_all[
            df_all['Defocus'] == defocus
        ].copy()
    )

    # Z_true from CUSTOM method
    z_true = (
        df_defocus[
            df_defocus['Method'] == 'CUSTOM'
        ]['Z_Final'].mean()
    )

    for method in methods:

        df_m = (
            df_defocus[
                df_defocus['Method'] == method
            ].copy()
        )

        if len(df_m) == 0:
            continue

        m_z = df_m['Z_Final'].mean()

        cv = (
            df_m['Z_Final'].std() / m_z
        ) * 100 if m_z != 0 else 0

        abs_step_diff = np.abs(
            df_m['Z_Final'] - z_true
        )

        rel_err_pct = (
            abs_step_diff / z_true * 100
        ).mean()

        all_metrics.append({

            'Method': method,
            'Defocus': defocus,
            'CV': cv,
            'Relative_Error': rel_err_pct,
            'Initial_Variance': df_m['Initial_Variance'].mean(),
            'Initial_Variance_SD': df_m['Initial_Variance'].std(),
            'Final_Variance': df_m['Final_Variance'].mean(),
            'Final_Variance_SD': df_m['Final_Variance'].std(),
            'Autofocus_Time': df_m['Autofocus_Time'].mean(),
            'Autofocus_Time_SD': df_m['Autofocus_Time'].std(),
            'CPU': df_m['CPU'].mean(),
            'CPU_SD': df_m['CPU'].std(),
            'RAM': df_m['RAM'].mean(),
            'RAM_SD': df_m['RAM'].std()
        })


df_metrics = pd.DataFrame(all_metrics)


# ============================================================
# 5. COLORS
# ============================================================

colors = {
    'SVR': '#1f77b4',
    'RANDOM FOREST': '#ff7f0e',
    'CUSTOM': '#2ca02c',
    'INBUILT': '#d62728'
}


# ============================================================
# 6. INDIVIDUAL DEFOCUS PLOTS
# ============================================================

for defocus in defocus_values:

    d_df = df_metrics[
        df_metrics['Defocus'] == defocus
    ]

    # AUTOFOCUS TIME
    fig, ax = plt.subplots(figsize=FIG_SINGLE, constrained_layout=True)
    bars = []
    for i, m in enumerate(methods):
        row = d_df[d_df['Method'] == m]
        if len(row) > 0:
            val = row['Autofocus_Time'].values[0]
            err = row['Autofocus_Time_SD'].values[0]
            if pd.isna(err):
                err = 0
            bar = ax.bar(i, val, yerr=err, color=colors[m], edgecolor='black', capsize=6, zorder=3, width=0.6, label=m)
            bars.append(bar)

    ax.set_xticks(range(len(methods)))
    ax.set_xticklabels(methods, rotation=15, fontweight='bold')
    format_axis(ax, ylabel='Autofocus Time (s)', title=f'Autofocus Time (Defocus = {defocus} Steps)')
    ax.grid(axis='y', linestyle='--', zorder=0)
    ax.legend(loc='best', fontsize=LEGEND_SIZE)
    for bar in bars:
        add_value_labels(ax, bar, fmt='{:.2f}')
    save_figure(fig, f"{out_dir}/Time_Defocus_{defocus}.png")

    # INITIAL VS FINAL VARIANCE
    fig, ax = plt.subplots(figsize=FIG_SINGLE, constrained_layout=True)
    width = 0.35
    iv_bars, fv_bars = [], []
    for i, m in enumerate(methods):
        row = d_df[d_df['Method'] == m]
        if len(row) > 0:
            iv = row['Initial_Variance'].values[0]
            iv_sd = row['Initial_Variance_SD'].values[0]
            fv = row['Final_Variance'].values[0]
            fv_sd = row['Final_Variance_SD'].values[0]

            bar1 = ax.bar(i - width / 2, iv, width, yerr=iv_sd if pd.notna(iv_sd) else 0, color=colors[m], edgecolor='black', capsize=6, hatch='//')
            iv_bars.append(bar1)
            bar2 = ax.bar(i + width / 2, fv, width, yerr=fv_sd if pd.notna(fv_sd) else 0, color=colors[m], edgecolor='black', capsize=6)
            fv_bars.append(bar2)

    ax.set_xticks(range(len(methods)))
    ax.set_xticklabels(methods, rotation=15, fontweight='bold')
    format_axis(ax, ylabel='Focus Quality (a.u.)', title=f'Initial vs Final Variance (Defocus = {defocus} Steps)')
    ax.grid(axis='y', linestyle='--', zorder=0)
    
    method_patches = [Patch(facecolor=colors[m], edgecolor='black', label=m) for m in methods]
    var_patches = [
        Patch(facecolor='white', edgecolor='black', hatch='//', label='Initial Variance'),
        Patch(facecolor='white', edgecolor='black', label='Final Variance')
    ]
    ax.legend(handles=method_patches + var_patches, loc='best', fontsize=LEGEND_SIZE)

    for b in iv_bars:
        add_value_labels(ax, b, fmt='{:.0f}', fontsize=VALUE_LABEL_SIZE)
    for b in fv_bars:
        add_value_labels(ax, b, fmt='{:.0f}', fontsize=VALUE_LABEL_SIZE)
    save_figure(fig, f"{out_dir}/Variance_Defocus_{defocus}.png")

    # CPU USAGE
    fig, ax = plt.subplots(figsize=FIG_SINGLE, constrained_layout=True)
    bars = []
    for i, m in enumerate(methods):
        row = d_df[d_df['Method'] == m]
        if len(row) > 0:
            val = row['CPU'].values[0]
            err = row['CPU_SD'].values[0]
            if pd.isna(err):
                err = 0
            bar = ax.bar(i, val, yerr=err, color=colors[m], edgecolor='black', capsize=6, zorder=3, width=0.6, label=m)
            bars.append(bar)

    ax.set_xticks(range(len(methods)))
    ax.set_xticklabels(methods, rotation=15, fontweight='bold')
    format_axis(ax, ylabel='CPU Usage (%)', title=f'CPU Usage (Defocus = {defocus} Steps)')
    ax.grid(axis='y', linestyle='--', zorder=0)
    ax.legend(loc='best', fontsize=LEGEND_SIZE)
    for bar in bars:
        add_value_labels(ax, bar, fmt='{:.1f}')
    save_figure(fig, f"{out_dir}/CPU_Defocus_{defocus}.png")

    # RAM USAGE (No error bars)
    fig, ax = plt.subplots(figsize=FIG_SINGLE, constrained_layout=True)
    bars = []
    for i, m in enumerate(methods):
        row = d_df[d_df['Method'] == m]
        if len(row) > 0:
            val = row['RAM'].values[0]
            bar = ax.bar(i, val, color=colors[m], edgecolor='black', zorder=3, width=0.6, label=m)
            bars.append(bar)

    ax.set_xticks(range(len(methods)))
    ax.set_xticklabels(methods, rotation=15, fontweight='bold')
    format_axis(ax, ylabel='RAM Usage (MB)', title=f'RAM Usage (Defocus = {defocus} Steps)')
    ax.grid(axis='y', linestyle='--', zorder=0)
    ax.legend(loc='best', fontsize=LEGEND_SIZE)
    for bar in bars:
        add_value_labels(ax, bar, fmt='{:.1f}')
    save_figure(fig, f"{out_dir}/RAM_Defocus_{defocus}.png")

    # CV AND RELATIVE ERROR
    metric_list = [
        ('CV', 'Coefficient of Variation (%)', f'CV (Defocus = {defocus} Steps)'),
        ('Relative_Error', 'Relative Error (%)', f'Relative Error (Defocus = {defocus} Steps)')
    ]
    for col, ylabel, title in metric_list:
        fig, ax = plt.subplots(figsize=FIG_SINGLE, constrained_layout=True)
        bars = []
        for i, m in enumerate(methods):
            row = d_df[d_df['Method'] == m]
            if len(row) > 0:
                val = row[col].values[0]
                bar = ax.bar(i, val, color=colors[m], edgecolor='black', zorder=3, width=0.6, label=m)
                bars.append(bar)

        ax.set_xticks(range(len(methods)))
        ax.set_xticklabels(methods, rotation=15, fontweight='bold')
        format_axis(ax, ylabel=ylabel, title=title)
        ax.grid(axis='y', linestyle='--', zorder=0)
        ax.legend(loc='best', fontsize=LEGEND_SIZE)
        for bar in bars:
            add_value_labels(ax, bar, fmt='{:.1f}')
        save_figure(fig, f"{out_dir}/{col}_Defocus_{defocus}.png")


# ============================================================
# 7. COMBINED BAR CHARTS
# ============================================================

metrics_to_combine = [
    ('CV', 'Coefficient of Variation (%)', 'CV across Defocus', True),
    ('Relative_Error', 'Relative Error (%)', 'Relative Error across Defocus', True),
    ('Autofocus_Time', 'Autofocus Time (s)', 'Autofocus Time across Defocus', False),
    ('CPU', 'CPU Utilization (%)', 'CPU Usage across Defocus', False),
    ('RAM', 'RAM Utilization (MB)', 'RAM Usage across Defocus', True),  # True removes error bars
    ('Initial_Variance', 'Initial Variance (a.u.)', 'Initial Variance across Defocus', False),
    ('Final_Variance', 'Final Variance (a.u.)', 'Final Variance across Defocus', False)
]

for col, ylabel, title, no_err in metrics_to_combine:
    fig, ax = plt.subplots(figsize=FIG_COMBINED, constrained_layout=True)
    x = np.arange(len(defocus_values))
    width = 0.2
    multiplier = 0

    err_col = None
    if col == 'Autofocus_Time':
        err_col = 'Autofocus_Time_SD'
    elif col == 'CPU':
        err_col = 'CPU_SD'
    elif col == 'Initial_Variance':
        err_col = 'Initial_Variance_SD'
    elif col == 'Final_Variance':
        err_col = 'Final_Variance_SD'

    for method in methods:
        vals = []
        errs = []
        for d in defocus_values:
            row = df_metrics[(df_metrics['Method'] == method) & (df_metrics['Defocus'] == d)]
            if len(row) > 0:
                vals.append(row[col].values[0])
                if err_col is not None and not no_err:
                    error_value = row[err_col].values[0]
                    errs.append(error_value if pd.notna(error_value) else 0)
                else:
                    errs.append(None)
            else:
                vals.append(np.nan)
                errs.append(None)

        offset = width * multiplier
        if no_err or all(e is None for e in errs):
            bars = ax.bar(x + offset, vals, width, label=method, color=colors[method], edgecolor='black', zorder=3)
        else:
            vals_masked = np.ma.masked_invalid(vals)
            errs_masked = np.ma.masked_invalid(errs)
            bars = ax.bar(x + offset, vals_masked, width, yerr=errs_masked, color=colors[method], edgecolor='black', capsize=5, zorder=3, label=method)

        multiplier += 1
        for bar in bars:
            height = bar.get_height()
            if not np.isnan(height):
                label = f'{height:.2f}' if height > 1 else f'{height:.1f}'
                ax.text(bar.get_x() + bar.get_width() / 2., height + 0.02 * abs(height) + 0.2, label, ha='center', va='bottom', fontsize=VALUE_LABEL_SIZE, fontweight='bold')

    ax.set_xticks(x + width * (len(methods) - 1) / 2)
    ax.set_xticklabels(defocus_values, fontweight='bold')
    format_axis(ax, xlabel='Defocus Steps', ylabel=ylabel, title=title)
    ax.legend(loc='best', fontsize=LEGEND_SIZE)
    ax.grid(axis='y', linestyle='--', zorder=0)
    save_figure(fig, f"{out_dir}/Combined_{col}_bar.png")


# ============================================================
# 8. TWO-PANEL: CHANGE IN SHARPNESS + RELATIVE ERROR
# ============================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=FIG_TWO_PANEL, constrained_layout=True)
width = 0.18

for method in methods:
    diffs = []
    for d in defocus_values:
        row = df_metrics[(df_metrics['Method'] == method) & (df_metrics['Defocus'] == d)]
        if len(row) > 0:
            diffs.append(row['Final_Variance'].values[0] - row['Initial_Variance'].values[0])
        else:
            diffs.append(np.nan)
    x = np.arange(len(defocus_values)) + 0.2 * (methods.index(method) - 1.5)
    bars = ax1.bar(x, diffs, width, label=method, color=colors[method], edgecolor='black')
    for bar in bars:
        h = bar.get_height()
        if not np.isnan(h):
            ax1.text(bar.get_x() + bar.get_width() / 2., h + 0.02 * abs(h) + 0.5, f'{h:.0f}', ha='center', va='bottom', fontsize=VALUE_LABEL_SIZE, fontweight='bold')

ax1.set_xticks(np.arange(len(defocus_values)))
ax1.set_xticklabels(defocus_values, fontweight='bold')
format_axis(ax1, xlabel='Defocus Steps', ylabel='Change in Sharpness (Final - Initial)', title='Change in Image Sharpness')
ax1.legend(loc='best', fontsize=LEGEND_SIZE)
ax1.grid(axis='y', linestyle='--', zorder=0)

for method in methods:
    vals = []
    for d in defocus_values:
        row = df_metrics[(df_metrics['Method'] == method) & (df_metrics['Defocus'] == d)]
        vals.append(row['Relative_Error'].values[0] if len(row) > 0 else np.nan)
    x = np.arange(len(defocus_values)) + 0.2 * (methods.index(method) - 1.5)
    bars = ax2.bar(x, vals, width, label=method, color=colors[method], edgecolor='black')
    for bar in bars:
        h = bar.get_height()
        if not np.isnan(h):
            ax2.text(bar.get_x() + bar.get_width() / 2., h + 0.02 * abs(h) + 0.2, f'{h:.2f}', ha='center', va='bottom', fontsize=VALUE_LABEL_SIZE, fontweight='bold')

ax2.set_xticks(np.arange(len(defocus_values)))
ax2.set_xticklabels(defocus_values, fontweight='bold')
format_axis(ax2, xlabel='Defocus Steps', ylabel='Relative Error (%)', title='Relative Error across Defocus')
ax2.legend(loc='best', fontsize=LEGEND_SIZE)
ax2.grid(axis='y', linestyle='--', zorder=0)
save_figure(fig, f"{out_dir}/Sharpness_RelativeError_combined.png")


# ============================================================
# 9. TWO-PANEL: AUTOFOCUS TIME + CV
# ============================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=FIG_TWO_PANEL, constrained_layout=True)

for method in methods:
    vals, errs = [], []
    for d in defocus_values:
        row = df_metrics[(df_metrics['Method'] == method) & (df_metrics['Defocus'] == d)]
        if len(row) > 0:
            vals.append(row['Autofocus_Time'].values[0])
            err = row['Autofocus_Time_SD'].values[0]
            errs.append(err if pd.notna(err) else 0)
        else:
            vals.append(np.nan)
            errs.append(0)
    x = np.arange(len(defocus_values)) + 0.2 * (methods.index(method) - 1.5)
    bars = ax1.bar(x, vals, width, yerr=errs, capsize=5, label=method, color=colors[method], edgecolor='black')
    for bar in bars:
        h = bar.get_height()
        if not np.isnan(h):
            ax1.text(bar.get_x() + bar.get_width() / 2., h + 0.02 * abs(h) + 0.5, f'{h:.2f}', ha='center', va='bottom', fontsize=VALUE_LABEL_SIZE, fontweight='bold')

ax1.set_xticks(np.arange(len(defocus_values)))
ax1.set_xticklabels(defocus_values, fontweight='bold')
format_axis(ax1, xlabel='Defocus Steps', ylabel='Autofocus Time (s)', title='Autofocus Time')
ax1.legend(loc='best', fontsize=LEGEND_SIZE)
ax1.grid(axis='y', linestyle='--', zorder=0)

for method in methods:
    vals = []
    for d in defocus_values:
        row = df_metrics[(df_metrics['Method'] == method) & (df_metrics['Defocus'] == d)]
        vals.append(row['CV'].values[0] if len(row) > 0 else np.nan)
    x = np.arange(len(defocus_values)) + 0.2 * (methods.index(method) - 1.5)
    bars = ax2.bar(x, vals, width, label=method, color=colors[method], edgecolor='black')
    for bar in bars:
        h = bar.get_height()
        if not np.isnan(h):
            ax2.text(bar.get_x() + bar.get_width() / 2., h + 0.02 * abs(h) + 0.2, f'{h:.2f}', ha='center', va='bottom', fontsize=VALUE_LABEL_SIZE, fontweight='bold')

ax2.set_xticks(np.arange(len(defocus_values)))
ax2.set_xticklabels(defocus_values, fontweight='bold')
format_axis(ax2, xlabel='Defocus Steps', ylabel='Coefficient of Variation (%)', title='CV across Defocus')
ax2.legend(loc='best', fontsize=LEGEND_SIZE)
ax2.grid(axis='y', linestyle='--', zorder=0)
save_figure(fig, f"{out_dir}/Time_CV_combined.png")


# ============================================================
# 10. TWO-PANEL: CPU + RAM
# ============================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=FIG_TWO_PANEL, constrained_layout=True)

for method in methods:
    vals, errs = [], []
    for d in defocus_values:
        row = df_metrics[(df_metrics['Method'] == method) & (df_metrics['Defocus'] == d)]
        if len(row) > 0:
            vals.append(row['CPU'].values[0])
            err = row['CPU_SD'].values[0]
            errs.append(err if pd.notna(err) else 0)
        else:
            vals.append(np.nan)
            errs.append(0)
    x = np.arange(len(defocus_values)) + 0.2 * (methods.index(method) - 1.5)
    bars = ax1.bar(x, vals, width, yerr=errs, capsize=5, label=method, color=colors[method], edgecolor='black')
    for bar in bars:
        h = bar.get_height()
        if not np.isnan(h):
            ax1.text(bar.get_x() + bar.get_width() / 2., h + 0.02 * abs(h) + 0.5, f'{h:.1f}', ha='center', va='bottom', fontsize=VALUE_LABEL_SIZE, fontweight='bold')

ax1.set_xticks(np.arange(len(defocus_values)))
ax1.set_xticklabels(defocus_values, fontweight='bold')
format_axis(ax1, xlabel='Defocus Steps', ylabel='CPU Utilization (%)', title='CPU Utilization')
ax1.legend(loc='best', fontsize=LEGEND_SIZE)
ax1.grid(axis='y', linestyle='--', zorder=0)

for method in methods:
    vals = []
    for d in defocus_values:
        row = df_metrics[(df_metrics['Method'] == method) & (df_metrics['Defocus'] == d)]
        vals.append(row['RAM'].values[0] if len(row) > 0 else np.nan)
    x = np.arange(len(defocus_values)) + 0.2 * (methods.index(method) - 1.5)
    bars = ax2.bar(x, vals, width, label=method, color=colors[method], edgecolor='black')
    for bar in bars:
        h = bar.get_height()
        if not np.isnan(h):
            ax2.text(bar.get_x() + bar.get_width() / 2., h + 0.02 * abs(h) + 0.5, f'{h:.1f}', ha='center', va='bottom', fontsize=VALUE_LABEL_SIZE, fontweight='bold')

ax2.set_xticks(np.arange(len(defocus_values)))
ax2.set_xticklabels(defocus_values, fontweight='bold')
format_axis(ax2, xlabel='Defocus Steps', ylabel='RAM Utilization (MB)', title='RAM Utilization')
ax2.legend(loc='best', fontsize=LEGEND_SIZE)
ax2.grid(axis='y', linestyle='--', zorder=0)
save_figure(fig, f"{out_dir}/CPU_RAM_combined.png")


# ============================================================
# COMPLETION MESSAGE
# ============================================================

print()
print("=" * 65)
print("ALL PRESENTATION-READY PLOTS HAVE BEEN GENERATED")
print("=" * 65)
print()
print(f"Output folder: {out_dir}")
print()
print("Key updates applied:")

print("  • Legends/keys included across all individual and combined plots.")
print("  • All figures configured for 600 DPI rendering.")
print("=" * 65)